# Week 49, Day 2: Hubs, modules and functional connectivity

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w049/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Spectral community detection

Write modQ(A, c) from the formula, and split(A, idx), which builds the modularity matrix restricted to the nodes idx (subtracting each row's sum from the diagonal, as Newman's generalised split requires) and divides them by the sign of its leading eigenvector. Split the network in two, then split each half.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import permutations
from scipy.sparse.csgraph import shortest_path
r = np.random.default_rng(2009)
M, n = 4, 30; N = M * n; lab = np.repeat(np.arange(M), n)
P = np.where(lab[:, None] == lab[None], 0.3, 0.02)
A = np.triu(r.random((N, N)) < P, 1); A = (A | A.T).astype(float)

def modQ(A, c):
    ...

def split(A, idx):
    k = A.sum(1); B = A - np.outer(k, k) / k.sum()
    Bg = B[np.ix_(idx, idx)]; Bg = Bg - np.diag(Bg.sum(1))
    ...

c = np.zeros(N, int); g = 0
for part in split(A, np.arange(N)):
    p1, p2 = split(A, part); c[p1] = g; c[p2] = g + 1; g += 2
acc = max(np.mean(np.array(perm)[c] == lab) for perm in permutations(range(4)))
print(f"modularity found {modQ(A, c):.3f}, planted {modQ(A, lab):.3f}; nodes correctly assigned {acc:.3f}")
order = np.argsort(c); plt.imshow(A[np.ix_(order, order)], cmap="gray_r"); plt.title("adjacency, sorted by found module"); plt.show()

In [ ]:
import numpy as np
def test_accuracy():
    "at least 85% of nodes are assigned to their planted community"
    assert acc > 0.85
def test_q():
    "the found partition has modularity above 0.4"
    assert modQ(A, c) > 0.4

test_accuracy()
test_q()
print("All checks passed")

### Exercise: Targeted attack against random failure

Write efficiency(A): the mean of 1 / shortest-path length over all ordered pairs of distinct nodes (unreachable pairs contribute 0). Remove the 20 highest-degree nodes, or 20 random nodes, and compare efficiency with the intact network.

In [ ]:
def grow(N=200, m=2):
    B = np.zeros((N, N)); B[0, 1] = B[1, 0] = 1
    for i in range(2, N):
        kk = B[:i, :i].sum(1) + 1e-9
        tg = r.choice(i, size=min(m, i), replace=False, p=kk / kk.sum()); B[i, tg] = B[tg, i] = 1
    return B

def efficiency(A):
    ...

B = grow(); k = B.sum(1); E0 = efficiency(B)
keep = lambda rm: np.setdiff1d(np.arange(len(B)), rm)
top, rnd = np.argsort(k)[::-1][:20], r.choice(len(B), 20, replace=False)
eff_targeted = efficiency(B[np.ix_(keep(top), keep(top))]) / E0
eff_random = efficiency(B[np.ix_(keep(rnd), keep(rnd))]) / E0
print(f"degrees: max {k.max():.0f}, mean {k.mean():.1f}; efficiency kept after random removal {eff_random:.2f}, after removing hubs {eff_targeted:.2f}")

In [ ]:
import numpy as np
def test_hubs():
    "the largest hub has more than five times the mean degree"
    assert k.max() > 5 * k.mean()
def test_attack():
    "removing hubs costs at least five times as much efficiency as removing random nodes"
    assert (1 - eff_targeted) > 5 * (1 - eff_random)

test_hubs()
test_attack()
print("All checks passed")

### Exercise: Indirect correlation disappears

Generate 20,000 samples of a = noise, b = 0.8 a + noise, c = 0.8 b + noise, d = noise. Compute the correlation matrix and the partial correlation matrix from the inverse covariance.

In [ ]:
T = 20000
a = r.normal(size=T); b = 0.8 * a + r.normal(size=T); c_ = 0.8 * b + r.normal(size=T); d = r.normal(size=T)
X = np.stack([a, b, c_, d], 1)
Cc = np.corrcoef(X.T)
Theta = ...
pc = ...
np.fill_diagonal(pc, 1)
print("correlation:\n", np.round(Cc, 2)); print("partial correlation:\n", np.round(pc, 2))

In [ ]:
import numpy as np
def test_corr():
    "A and C are substantially correlated (above 0.3) although not connected"
    assert Cc[0, 2] > 0.3
def test_partial():
    "their partial correlation is near zero, while A-B and B-C remain strong"
    assert abs(pc[0, 2]) < 0.05 and pc[0, 1] > 0.3 and pc[1, 2] > 0.3

test_corr()
test_partial()
print("All checks passed")